# Modelado Predictivo - Riesgo de Crédito

En este cuaderno construiremos un modelo baseline para predecir la probabilidad de incumplimiento (`target_bad`). 
Utilizaremos un pipeline de **Regresión Logística** que incluye imputación de nulos, escalado de variables numéricas y codificación de variables categóricas.

## 1. Importación de Librerías

Cargamos las herramientas necesarias: `pandas` para datos, `sklearn` para modelado y métricas.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    recall_score,
    precision_score,
    roc_auc_score,
    average_precision_score
)

## 2. Carga del Dataset Procesado

Leemos el archivo `data/df_sample.csv` que generamos en la etapa de EDA.

In [ ]:

df_sample = pd.read_csv("../data/df_sample.csv")
print(df_sample.shape)
df_sample.head()

## 3. Definición de Variables (Features y Target)

Reconstruimos las listas de columnas para segregar: 
*   `target_col`: La variable a predecir.
*   `leak_cols`, `id_text_cols`: Variables a excluir para evitar fugas de información o ruido.
*   `X_cols`: Las variables predictoras finales.

In [ ]:
# 1) Target
if "target_bad" in df_sample.columns:
    target_col = "target_bad"
else:
    raise NameError(
        "No existe target_bad en df_sample. Regresa a 01_eda, crea target_bad y vuelve a guardar data/df_sample.csv"
    )

# 2) Columnas que no deben entrar al modelo
id_text_cols = []
for c in ["id", "member_id", "url", "desc", "emp_title", "title"]:
    if c in df_sample.columns:
        id_text_cols.append(c)

leak_candidates = [
    "loan_status",
    "total_pymnt", "total_pymnt_inv",
    "total_rec_prncp", "total_rec_int", "total_rec_late_fee",
    "recoveries", "collection_recovery_fee",
    "last_pymnt_d", "last_pymnt_amnt",
    "next_pymnt_d",
    "last_credit_pull_d",
    "out_prncp", "out_prncp_inv",
]

leak_cols = [c for c in leak_candidates if c in df_sample.columns]

optional_cols = []
base_cols = [c for c in df_sample.columns if c not in ([target_col] + id_text_cols + leak_cols)]
X_cols = base_cols + optional_cols

print("target_col:", target_col)
print("id_text_cols:", len(id_text_cols), id_text_cols)
print("leak_cols:", len(leak_cols), leak_cols)
print("X_cols:", len(X_cols))

df_sample[target_col].value_counts(dropna=False)

## 4. Limpieza Final y División Train/Test

1.  Eliminamos filas donde el target sea nulo (no podemos entrenar ni validar con ellas).
2.  Eliminamos columnas que están 100% vacías.
3.  Dividimos los datos en conjuntos de Entrenamiento (80%) y Prueba (20%) usando muestreo estratificado para mantener la proporción de clases.

In [ ]:
RANDOM_STATE = 12345

# Dataset para modelar sin NaN en target
df_model = df_sample.dropna(subset=[target_col]).copy()

# Detecta columnas totalmente vacías
cols_all_nan = [c for c in X_cols if df_model[c].isna().all()]
print("Columnas 100% NaN:", len(cols_all_nan))
print(cols_all_nan)

# Actualiza lista de features
X_cols_clean = [c for c in X_cols if c not in cols_all_nan]
print("X_cols antes:", len(X_cols))
print("X_cols después:", len(X_cols_clean))

# Split
X = df_model[X_cols_clean].copy()
y = df_model[target_col].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Distribución y_train")
print(y_train.value_counts(normalize=True).round(4))

## 5. Construcción del Pipeline y Entrenamiento

Definimos un pipeline de procesamiento que maneja automáticamente:
*   **Variables Numéricas:** Imputación de nulos con la mediana + Estandarización.
*   **Variables Categóricas:** Imputación con la moda + One Hot Encoding.

El modelo elegido es una **Regresión Logística** con balanceo de clases (`class_weight="balanced"`) para penalizar más los errores en la clase minoritaria (los malos pagadores).

In [ ]:
# Preprocesamiento y modelo

num_features = X_train.select_dtypes(include=["number"]).columns.tolist()
cat_features = [c for c in X_train.columns if c not in num_features]

print("Num features:", len(num_features))
print("Cat features:", len(cat_features))

num_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=True)

cat_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", ohe)
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_features),
        ("cat", cat_pipe, cat_features)
    ],
    remainder="drop"
)

model = LogisticRegression(
    max_iter=6000,
    tol=1e-3,
    solver="saga",
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

clf = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", model)
    ]
)

clf.fit(X_train, y_train)

proba_test = clf.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= 0.5).astype(int)

print("Listo, modelo entrenado")

## 6. Evaluación de Métricas

Evaluamos el rendimiento del modelo en el conjunto de prueba.
*   **Recall (Sensibilidad):** Qué tan bien detectamos a los malos pagadores.
*   **Precision:** De los que etiquetamos como malos, ¿cuántos realmente lo eran?
*   **ROC AUC:** Capacidad global de discriminar entre clases.
*   **Métricas de Negocio:** Simulamos el impacto financiero revisando cuántos malos se aprobarían (Falsos Negativos).

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    recall_score,
    precision_score,
    roc_auc_score,
    average_precision_score
)

cm = confusion_matrix(y_test, pred_test, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

recall_1 = recall_score(y_test, pred_test, pos_label=1)
precision_1 = precision_score(y_test, pred_test, pos_label=1, zero_division=0)
roc_auc = roc_auc_score(y_test, proba_test)
pr_auc = average_precision_score(y_test, proba_test)

approved = (pred_test == 0).sum()
approval_rate = approved / len(pred_test)

bad_approved = fn
bad_approval_rate_among_approved = bad_approved / max(approved, 1)

print("Confusion matrix [0,1]")
print(cm)

print("\nMétricas clave")
print("Recall clase 1:", round(recall_1, 4))
print("Precision clase 1:", round(precision_1, 4))
print("ROC AUC:", round(roc_auc, 4))
print("PR AUC:", round(pr_auc, 4))

print("\nNegocio")
print("Approval rate:", round(approval_rate, 4))
print("Bad approvals (FN):", int(bad_approved))
print("Bad approval rate among approved:", round(bad_approval_rate_among_approved, 4))

print("\nReporte")
print(classification_report(y_test, pred_test, digits=4))

## 7. Optimización del Umbral de Decisión

El umbral por defecto es 0.5. Sin embargo, en riesgo de crédito a menudo queremos ser más conservadores. 
Generamos una tabla para inspeccionar cómo cambian las métricas (Tasa de aprobación vs Riesgo tolerado) al variar el umbral de aceptación.

In [ ]:
import pandas as pd

def metrics_at_threshold(y_true, proba, thr):
    pred = (proba >= thr).astype(int)
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    approved = (pred == 0).sum()
    total = len(pred)

    return {
        "threshold": float(thr),
        "recall_1": float(recall_score(y_true, pred, pos_label=1)),
        "precision_1": float(precision_score(y_true, pred, pos_label=1, zero_division=0)),
        "approval_rate": float(approved / total),
        "bad_approved": int(fn),
        "bad_approval_rate_among_approved": float(fn / max(approved, 1))
    }

thresholds = np.round(np.linspace(0.1, 0.9, 17), 2)
thr_table = pd.DataFrame([metrics_at_threshold(y_test, proba_test, t) for t in thresholds])

thr_table.sort_values(["bad_approved", "approval_rate"], ascending=[True, False]).head(10)

### Selección Automática del Umbral

Como ejemplo, filtramos aquellos umbrales que nos den una tasa de aprobación de al menos el 45% y elegimos el que minimice la cantidad de "malos aprobados".

In [ ]:
candidates = thr_table[thr_table["approval_rate"] >= 0.45]
print("Candidatos:", len(candidates))
if len(candidates) > 0:
    best_row = candidates.sort_values("bad_approved", ascending=True).iloc[0]
    print("Mejor umbral sugerido:")
    print(best_row)
else:
    print("Ningún umbral cumple la condición de aprobación mínima.")